# OpenDiffraction: Dataset Exploration

**OpenDiffraction: Physics-Based Synthetic Coherent Diffraction Patterns for Phase Retrieval and Image Reconstruction**

This notebook demonstrates how to:
1. Inspect the dataset manifest and metadata
2. Load HDF5 shards using `OpenDiffractionDataset`
3. Access ground-truth real-space object densities
4. Visualize raw photon counts vs. log-scaled diffraction intensity
5. Apply detector validity masks (beam stops, gaps, defect pixels)
6. Compare clean vs. noisy observations across different variation profiles (V01 - V12)

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import h5py

from opendiffraction.io.loader import OpenDiffractionDataset

# Set output directory
dataset_dir = "../output"
print(f"Inspecting dataset at: {os.path.abspath(dataset_dir)}")

## 1. Inspecting Metadata Manifest

The dataset includes a comprehensive `metadata.csv` recording experimental parameters, photon counts, coherence lengths, and object group identifiers.

In [ ]:
metadata_path = os.path.join(dataset_dir, "metadata.csv")
if os.path.exists(metadata_path):
    df = pd.read_csv(metadata_path)
    print(f"Total records in manifest: {len(df)}")
    print("\nSplit distribution:")
    print(df["split"].value_counts())
    print("\nVariation profile distribution:")
    print(df["variation_family"].value_counts().sort_index())
    display(df.head())
else:
    print("metadata.csv not yet generated. Run generation first.")

## 2. Loading Samples via Dataset API

We load a sample using the `OpenDiffractionDataset` API, which exposes both the DPR-compatible tensors and the scientific metadata.

In [ ]:
ds = OpenDiffractionDataset(dataset_dir)
print(f"Loaded dataset with {len(ds)} samples.")

# Load first sample
sample = ds[0]
print(f"Sample ID: {sample.sample_id}")
print(f"Object amplitude shape: {sample.object_amplitude.shape}, dtype: {sample.object_amplitude.dtype}")
print(f"Detector input shape: {sample.input.shape}, dtype: {sample.input.dtype}")
print(f"Mask shape: {sample.mask.shape}, dtype: {sample.mask.dtype}")

## 3. Visualizing Object vs. Diffraction Intensity

We compare the unscaled real-space object against:
- Raw linear photon count measurement
- Log-scaled display intensity $\log(1 + \max(I, 0))$
- Detector validity mask and masked observation

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 1. Real-space object
axes[0].imshow(sample.object_amplitude[0], cmap="inferno", origin="lower")
axes[0].set_title("Ground Truth $O(r)$")
axes[0].axis("off")

# 2. Linear raw readout (dynamic range compresses fringes)
axes[1].imshow(sample.input[0], cmap="viridis", origin="lower")
axes[1].set_title("Raw Linear Counts")
axes[1].axis("off")

# 3. Log-transformed intensity
log_I = np.log1p(np.maximum(sample.input[0], 0))
axes[2].imshow(log_I, cmap="viridis", origin="lower")
axes[2].set_title("Log Intensity $\\log(1 + I)$")
axes[2].axis("off")

# 4. Masked observation
obs_masked = sample.input[0] * sample.mask[0].astype(np.float32)
axes[3].imshow(np.log1p(np.maximum(obs_masked, 0)), cmap="viridis", origin="lower")
axes[3].set_title("Masked Observation")
axes[3].axis("off")

plt.tight_layout()
plt.show()

## 4. Comparing Variation Families

Let's sample across different variation profiles: clean coherent (V01), high geometry diversity (V02), photon starvation (V04), and realistic combined degradation (V11).

In [ ]:
profiles_to_show = ["V01", "V02", "V04", "V05", "V08", "V11"]
fig, axes = plt.subplots(2, len(profiles_to_show), figsize=(18, 6))

for col, pid in enumerate(profiles_to_show):
    subset = OpenDiffractionDataset(dataset_dir, variation=pid)
    if len(subset) > 0:
        s = subset[0]
        # Row 0: Object
        axes[0, col].imshow(s.object_amplitude[0], cmap="inferno", origin="lower")
        axes[0, col].set_title(f"{pid}: Object")
        axes[0, col].axis("off")
        # Row 1: Log masked diffraction pattern
        obs = s.input[0] * s.mask[0].astype(np.float32)
        axes[1, col].imshow(np.log1p(np.maximum(obs, 0)), cmap="viridis", origin="lower")
        axes[1, col].set_title(f"{pid}: Diffraction")
        axes[1, col].axis("off")

plt.tight_layout()
plt.show()